#   Libraries

In [12]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
# Download stopwords list from NLTK if not already downloaded
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LOQ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#   Loading The Data & Text Cleaning

In [14]:
df = pd.read_csv('../data/UpdatedResumeDataSet.csv')
display(df.head())

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."


In [15]:
def clean_resume(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(r'http\S+\s*', ' ', text) # Remove URLs
    text = re.sub(r'#\S+', '', text) # Remove hashtags
    text = re.sub(r'@\S+', '  ', text) # Remove mentions
    text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text) # Remove punctuations
    text = re.sub(r'[^\x00-\x7f]', r' ', text) # Remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text) # Remove extra whitespaces
    
    # Remove stopwords (common words that add no meaning to the model)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

<>:6: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<>:6: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
C:\Users\LOQ\AppData\Local\Temp\ipykernel_8668\1459919544.py:6: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
  text = re.sub(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text) # Remove punctuations


In [16]:
# 3. Apply the cleaning function to the resume text column
df['Cleaned_Resume'] = df['Resume'].apply(clean_resume)

# Compare the original text with the cleaned text to see the difference
print("\n--- Original vs Cleaned Resume Example ---")
print("Original:", df['Resume'].iloc[0][:200], "...")
print("Cleaned:", df['Cleaned_Resume'].iloc[0][:200], "...")


--- Original vs Cleaned Resume Example ---
Original: Skills * Programming Languages: Python (pandas, numpy, scipy, scikit-learn, matplotlib), Sql, Java, JavaScript/JQuery. * Machine learning: Regression, SVM, NaÃ¯ve Bayes, KNN, Random Forest, Decision T ...
Cleaned: skills programming languages python pandas numpy scipy scikit learn matplotlib sql java javascript jquery machine learning regression svm na bayes knn random forest decision trees boosting techniques  ...


#   Vectorizing And Converting Text

In [17]:
# 1. Initialize TF-IDF Vectorizer to convert text into numerical vectors
vectorizer = TfidfVectorizer(max_features=5000)

In [18]:
# 2. Fit and transform the cleaned resumes into a matrix
resume_vectors = vectorizer.fit_transform(df['Cleaned_Resume'])

In [19]:
# 3. Simulate a Job Description provided by an HR manager
job_description = """
Looking for a Python Developer with strong experience in machine learning, 
data analysis, pandas, numpy, and natural language processing. Good communication 
skills and problem-solving abilities are required.
"""

In [20]:
# 4. Clean the Job Description using the same cleaning logic
# (Using the clean_resume function we defined earlier)
cleaned_jd = clean_resume(job_description)

In [21]:
# 5. Transform the Job Description using the SAME vectorizer
jd_vector = vectorizer.transform([cleaned_jd])

In [22]:
# 6. Calculate Cosine Similarity between the Job Description and all Resumes
similarity_scores = cosine_similarity(jd_vector, resume_vectors)

In [23]:
# 7. Get the top matching resumes
# flatten() converts the 2D array to 1D, and argsort() sorts indices from lowest to highest similarity
top_indices = similarity_scores.flatten().argsort()[::-1]

In [24]:
# Display the top 3 best matching resumes
print("--- Top 3 Matching Resumes for the Job Description ---")
for i in range(3):
    idx = top_indices[i]
    score = similarity_scores[0][idx] * 100
    category = df['Category'].iloc[idx] if 'Category' in df.columns else "General"
    print(f"Rank {i+1}: Resume ID {idx} | Category: {category} | Match Score: {score:.2f}%")

--- Top 3 Matching Resumes for the Job Description ---
Rank 1: Resume ID 17 | Category: Data Science | Match Score: 28.73%
Rank 2: Resume ID 37 | Category: Data Science | Match Score: 28.73%
Rank 3: Resume ID 27 | Category: Data Science | Match Score: 28.73%
